In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

# ==========================================
# CONFIGURACIÓN
# ==========================================
storage_account = "adsldevcrm"
container_name = "raw"
scope = "databriksScope"

client_id = dbutils.secrets.get(scope=scope, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope, key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope=scope, key="sp-tenant-id")

gold_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/gold"
database_name = "crm_sales_db"

In [0]:
# ==========================================
# AUTENTICACIÓN
# ==========================================
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

spark.sql(f"USE {database_name}")

In [0]:
# ==========================================
# LEER SILVER
# ==========================================
accounts_df = spark.table(f"{database_name}.silver_accounts")
products_df = spark.table(f"{database_name}.silver_products")
sales_pipeline_df = spark.table(f"{database_name}.silver_sales_pipeline")
sales_teams_df = spark.table(f"{database_name}.silver_sales_teams")

In [0]:
# ==========================================
# TABLA DETALLE
# ==========================================
sales_detail_df = sales_pipeline_df.alias("sp") \
    .join(accounts_df.alias("a"), F.col("sp.account") == F.col("a.account"), "left") \
    .join(products_df.alias("p"), F.col("sp.product") == F.col("p.product"), "left") \
    .join(sales_teams_df.alias("t"), F.col("sp.sales_agent") == F.col("t.sales_agent"), "left") \
    .select(
        F.col("sp.opportunity_id"),
        F.col("sp.sales_agent"),
        F.col("t.manager"),
        F.col("t.regional_office"),
        F.col("sp.product"),
        F.col("p.series"),
        F.col("sp.account"),
        F.col("a.sector"),
        F.col("a.office_location"),
        F.col("a.parent_company"),
        F.col("sp.deal_stage"),
        F.col("sp.engage_date"),
        F.col("sp.close_date"),
        F.col("sp.close_value")
    )

In [0]:
# ==========================================
# KPI POR ETAPA
# ==========================================
kpi_by_stage_df = sales_pipeline_df.groupBy("deal_stage").agg(
    F.count("*").alias("opportunities"),
    F.sum("close_value").alias("total_close_value"),
    F.avg("close_value").alias("avg_close_value")
)

# ==========================================
# VENTAS POR AGENTE
# ==========================================
sales_by_agent_df = sales_pipeline_df.alias("sp") \
    .join(sales_teams_df.alias("t"), F.col("sp.sales_agent") == F.col("t.sales_agent"), "left") \
    .groupBy("sp.sales_agent", "t.manager", "t.regional_office") \
    .agg(
        F.count("*").alias("opportunities"),
        F.sum(F.when(F.col("sp.deal_stage") == "Won", F.col("sp.close_value")).otherwise(0)).alias("won_value"),
        F.sum(F.when(F.col("sp.deal_stage") == "Won", 1).otherwise(0)).alias("won_count"),
        F.sum(F.when(F.col("sp.deal_stage") == "Lost", 1).otherwise(0)).alias("lost_count")
    )

# ==========================================
# VENTAS POR PRODUCTO
# ==========================================
sales_by_product_df = sales_pipeline_df.alias("sp") \
    .join(products_df.alias("p"), F.col("sp.product") == F.col("p.product"), "left") \
    .groupBy("sp.product", "p.series") \
    .agg(
        F.count("*").alias("opportunities"),
        F.sum(F.when(F.col("sp.deal_stage") == "Won", F.col("sp.close_value")).otherwise(0)).alias("won_value")
    )

# ==========================================
# VENTAS POR SECTOR
# ==========================================
sales_by_sector_df = sales_pipeline_df.alias("sp") \
    .join(accounts_df.alias("a"), F.col("sp.account") == F.col("a.account"), "left") \
    .groupBy("a.sector") \
    .agg(
        F.count("*").alias("opportunities"),
        F.sum(F.when(F.col("sp.deal_stage") == "Won", F.col("sp.close_value")).otherwise(0)).alias("won_value")
    )

# ==========================================
# TENDENCIA MENSUAL WON
# ==========================================
monthly_won_trend_df = sales_pipeline_df \
    .filter(F.col("deal_stage") == "Won") \
    .withColumn("close_year", F.year("close_date")) \
    .withColumn("close_month", F.month("close_date")) \
    .groupBy("close_year", "close_month") \
    .agg(
        F.count("*").alias("won_opportunities"),
        F.sum("close_value").alias("won_value")
    )

In [0]:
# ==========================================
# REVISIÓN
# ==========================================
display(sales_detail_df)
display(kpi_by_stage_df)
display(sales_by_agent_df)
display(sales_by_product_df)
display(sales_by_sector_df)
display(monthly_won_trend_df)

In [0]:
# ==========================================
# GUARDAR TABLAS GOLD
# ==========================================
sales_detail_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.gold_sales_detail")
kpi_by_stage_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.gold_kpi_by_stage")
sales_by_agent_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.gold_sales_by_agent")
sales_by_product_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.gold_sales_by_product")
sales_by_sector_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.gold_sales_by_sector")
monthly_won_trend_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{database_name}.gold_monthly_won_trend")

# ==========================================
# GUARDAR RUTAS GOLD
# ==========================================
sales_detail_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/sales_detail")
kpi_by_stage_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/kpi_by_stage")
sales_by_agent_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/sales_by_agent")
sales_by_product_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/sales_by_product")
sales_by_sector_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/sales_by_sector")
monthly_won_trend_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/monthly_won_trend")

print("Gold terminado")